We use TDC datasets for several classification and regression tasks mentioned in the preprint, using either random or scaffold splits with three different random seeds to produce triplicates.

# 01. Generate Features

In [ ]:
# pkgs
import h5py
import torch
import numpy as np

from pathlib import Path
from tqdm.notebook import tqdm
from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem, Crippen, Descriptors, Lipinski, MolSurf

# disable warnings
RDLogger.DisableLog('rdApp.*')

from intermol.main.sae import SparseAutoencoder
from intermol.main.utils import load_model_from_HF, load_model_from_file

# defaults
LAYERS = [1, 3, 6, 9, 12]
DIRS = [next(Path("..").glob(f"results_layer{layer}_*")) for layer in LAYERS]

DATA_DIR = Path("../data/tdc")
DATA_PTH = DATA_DIR / "cSMILES.txt"
OUTFN_PTH = DATA_DIR / "probing_features.h5"

DEVICE = torch.device("cuda" if torch.cuda.is_available else "cpu")

## SAE config
HIDDEN_DIM = 3072
K = 128

In [ ]:
# init data
## use extracted unique canonical SMILES across all TDC datasets
## to ensure correct SMILES ordering within 'OUTFN_PTH'
## when using TDC datasets split with different random seeds

with open(DATA_PTH, 'r') as h:
    data = [smi.rstrip("\n") for smi in h.readlines()]
n_data = len(data)

## 01A. FP & Physicochemical Descriptors

In [ ]:
# init FP & Physicochemical Descriptors
rdk_fs = {
    "ECFP0": AllChem.GetMorganGenerator(radius=0, fpSize=2048),
    "ECFP2": AllChem.GetMorganGenerator(radius=1, fpSize=2048),
    "ECFP4": AllChem.GetMorganGenerator(radius=2, fpSize=2048),
    "PcDs": {
        "LogP": Crippen.MolLogP,
        "MR": Crippen.MolMR,
        "MW": Descriptors.MolWt,
        "FormalCharge": Chem.GetFormalCharge,
        "FractionCSP3": Lipinski.FractionCSP3,
        "HeavyAtomCount": Lipinski.HeavyAtomCount,
        "HBA": Lipinski.NumHAcceptors,
        "HBD": Lipinski.NumHDonors,
        "Rotatables": Lipinski.NumRotatableBonds,
        "RingCount": Lipinski.RingCount,
        "TPSA": MolSurf.TPSA,
        "AtomCount": lambda mol: mol.GetNumAtoms(onlyExplicit=False)
    }
}

# generate features
with h5py.File(OUTFN_PTH, 'a') as out_h5f:
    # RDKit-parsable checks
    is_parsable = np.fromiter(
        (Chem.MolFromSmiles(smi) is not None for smi in data),
        dtype=np.uint8,
        count=len(data)
    )
    out_h5f.create_dataset(
        "isSmilesRDKitParsable",
        data=is_parsable,
        compression="lzf"
    )

    # FP & Physicochemical
    for f_name, rdk_gen in tqdm(rdk_fs.items(), desc="Generating features..."):
        if f_name == "PcDs":
            arr = np.zeros((n_data, 12))
            for smi_i, smi in enumerate(data):
                mol = Chem.MolFromSmiles(smi)
                if mol:
                    arr[smi_i, :] = [gen(mol) for gen in rdk_gen.values()]
        else:
            arr = np.zeros((n_data, 2048), dtype=np.uint8)
            for smi_i, smi in enumerate(data):
                mol = Chem.MolFromSmiles(smi)
                if mol:
                    arr[smi_i, :] = rdk_gen.GetFingerprintAsNumPy(mol)

        # create dataset
        out_h5f.create_dataset(f_name, data=arr, compression="lzf")

    # clean up
    del is_parsable, arr

## 01B. MolFormer & SAE Embeddings

In [ ]:
# init MolFormer and SAE
tokenizer, mf_model = load_model_from_HF(model_name="ibm/MoLFormer-XL-both-10pct")
mf_model = mf_model.to(DEVICE)
mf_hidden_dim = mf_model.config.hidden_size

sae_models = {
    layer: load_model_from_file(
        SparseAutoencoder(hidden_dim=HIDDEN_DIM, k=K),
        file_pth=DIRS[l_i] / "norm-sae_state-dict_273908s_12165077t.pt"
    ).to(DEVICE) for l_i, layer in enumerate(LAYERS)
}

# inference
mf_shape = (len(LAYERS), n_data, mf_hidden_dim)
mean_mf = torch.zeros(mf_shape, dtype=torch.float32)
max_mf = torch.zeros(mf_shape, dtype=torch.float32)

sae_shape = (len(LAYERS), n_data, HIDDEN_DIM)
mean_sae = torch.zeros(sae_shape, dtype=torch.float32)
max_sae = torch.zeros(sae_shape, dtype=torch.float32)

with torch.no_grad():
    for smi_i, smi in tqdm(enumerate(data), desc="Generating features..."):
        enc = tokenizer(smi).to(DEVICE)
        mf_acts = mf_model(**enc, output_hidden_states=True).hidden_states
        for l_i, layer in enumerate(LAYERS):
            mf_act = mf_acts[layer].squeeze()[1:-1, :]
            mean_mf[l_i, smi_i, :] = mf_act.mean(dim=0)
            max_mf[l_i, smi_i, :] = mf_act.max(dim=0).values

            sae_act = (
                sae_models[layer]
                .encode_latents(mf_act[layer])
                .squeeze()[1:-1, :]
            )
            mean_sae[l_i, smi_i, :] = sae_act.mean(dim=0)
            max_sae[l_i, smi_i, :] = sae_act.max(dim=0).values

del tokenizer, mf_model, sae_models, enc, mf_acts, mf_act, sae_act

# to numpy
mean_mf = mean_mf.cpu().numpy()
max_mf = max_mf.cpu().numpy()
mean_sae = mean_sae.cpu().numpy()
max_sae = max_sae.cpu().numpy()

# generate features
with h5py.File(OUTFN_PTH, 'a') as out_h5f:
    # create groups
    g_mf = out_h5f.require_group("MolFormer")
    g_mf_mean = g_mf.require_group("Mean")
    g_mf_max = g_mf.require_group("Max")

    g_sae = out_h5f.require_group("SAE")
    g_sae_mean = g_sae.require_group("Mean")
    g_sae_max = g_sae.require_group("Max")

    # create datasets
    for l_i, layer in enumerate(LAYERS):
        g_mf_mean.create_dataset(
            str(layer), data=mean_mf[l_i, :, :], compression="lzf"
        )
        g_mf_max.create_dataset(
            str(layer), data=max_mf[l_i, :, :], compression="lzf"
        )

        g_sae_mean.create_dataset(
            str(layer), data=mean_sae[l_i, :, :], compression="lzf"
        )
        g_sae_max.create_dataset(
            str(layer), data=max_sae[l_i, :, :], compression="lzf"
        )

del mean_mf, max_mf, mean_sae, max_sae

# 02. Linear Probing

In [ ]:
# pkgs
import polars as pl

from tqdm.notebook import tqdm
from sklearn.linear_model import LogisticRegressionCV, RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    make_scorer,
    accuracy_score,
    matthews_corrcoef,
    roc_auc_score,
    root_mean_squared_error
)
from scipy.stats import spearmanr

import warnings
# ignore warnings
warnings.filterwarnings('ignore')

# defaults
DATA_DIR = Path("../data/tdc")
FEATURES_PTH = DATA_DIR / "probing_features.h5"
OUTFN_PTH = DATA_DIR / "linear_probes_result.parquet"

FEATURE_GROUP = {
    "RDKit": ["ECFP0", "ECFP2", "ECFP4", "PcDs"],
    "MolFormer": [
        "MolFormer/Mean/1", "MolFormer/Mean/3", "MolFormer/Mean/6",
        "MolFormer/Mean/9", "MolFormer/Mean/12",
        "MolFormer/Max/1", "MolFormer/Max/3", "MolFormer/Max/6",
        "MolFormer/Max/9", "MolFormer/Max/12"
    ],
    "SAE": [
        "SAE/Mean/1", "SAE/Mean/3", "SAE/Mean/6", "SAE/Mean/9", "SAE/Mean/12",
        "SAE/Max/1", "SAE/Max/3", "SAE/Max/6", "SAE/Max/9", "SAE/Max/12"
    ]
}


# init data
with open(DATA_PTH, 'r') as h:
    data_map = {smi.rstrip("\n"): smi_i for smi_i, smi in enumerate(h.readlines())}
n_data = len(data_map)

In [ ]:
# probing latents
out = []
for d_i, data_pth in DATA_DIR.glob("dataset_*.parquet"):
    # parse dataset
    dataset = pl.read_parquet(data_pth)
    dataset = (
        dataset
        # aggregate duplicates
        .group_by([col for col in dataset.columns if col != "Y"])
        .agg(
            pl.when(pl.col("taskType").first() == "reg")
            .then(pl.col("Y").mean())
            .otherwise(pl.col("Y").max())
            .alias("Y")
        )
        # map cSMILES to their indices
        .with_columns(
            pl.col("cSMILES").replace_strict(data_map).alias("cIdx")
        )
    )

    # get available tasks
    tasks = dataset["task"].unique(maintain_order=True).to_list()

    # gather features
    with h5py.File(FEATURES_PTH, 'r') as out_h5f:
        # RDK features sanity check
        chk = out_h5f["isSMILESRDKitParsable"][:]
        chkptr = np.flatnonzero(chk == 0).tolist()

        if len(chkptr) > 0:
            warnings.warn(
                f"Found {len(chkptr)} RDKit-unparsable SMILES. "
                "These samples will be excluded from RDKit-based feature evaluation."
            )

        for fg, fs_name in tqdm(FEATURE_GROUP.items(), desc="Processing per group..."):
            for task in tqdm(tasks, desc="Processing per task...", leave=False):
                if fg == "RDKit":
                    subset = dataset.filter(
                        (pl.col("task") == task) &
                        (~pl.col("cIdx").is_in(chkptr))
                    ).sort("cIdx")
                else:
                    subset = dataset.filter(pl.col("task") == task).sort("cIdx")

                task_type = subset["taskType"].unique().item()
                label_dtype = np.int8 if task_type == "cls" else np.float64

                # gather train-test splits
                cond = (pl.col("split") == "test")
                subset_train = subset.filter(~cond)
                subset_test = subset.filter(cond)

                label_train = subset_train["Y"].to_numpy().astype(label_dtype)
                label_test = subset_test["Y"].to_numpy().astype(label_dtype)

                ptr_train = subset_train["Y"].to_numpy()
                ptr_test = subset_test["Y"].to_numpy()

                for f_name in fs_name:
                    split_train = out_h5f[f_name][ptr_train]
                    split_test = out_h5f[f_name][ptr_test]

                    # normalize X values for PcDs
                    if f_name == "PcDs":
                        scaler = StandardScaler()
                        split_train = scaler.fit_transform(split_train)
                        split_test = scaler.fit_transform(split_test)
                        del scaler

                    # model
                    if task_type == "cls":
                        model = LogisticRegressionCV(
                            scoring=make_scorer(roc_auc_score),
                            cv=5,
                            n_jobs=-1,
                            max_iter=1000
                        )
                        model.fit(split_train, label_train)

                        # test
                        y_pred = model.predict(split_test)
                        y_pred_proba = model.predict_proba(split_test)[:, 1]

                        out.append({
                            "Task": task,
                            "Group": fg,
                            "Method": f_name,
                            "Model": "Logistic Regression",
                            "Rep": d_i + 1,
                            "Params": model.C_[0],
                            "Coefs": model.coef_.flatten().tolist(),
                            "Intercept": model.intercept_[0],
                            "Accuracy": accuracy_score(label_test, y_pred),
                            "MCC": matthews_corrcoef(label_test, y_pred),
                            "ROC-AUC": roc_auc_score(label_test, y_pred_proba)
                        })

                    elif task_type == "reg":
                        model = RidgeCV(
                            alphas=(0.1, 1.0, 10.0, 100.0),
                            scoring="neg_root_mean_squared_error",
                            cv=5
                        )
                        model.fit(split_train, label_train)

                        # test
                        y_pred = model.predict(split_test)

                        out.append({
                            "Task": task,
                            "Group": fg,
                            "Method": f_name,
                            "Model": "Ridge Regression",
                            "Rep": d_i + 1,
                            "Params": model.alpha_,
                            "Coefs": model.coef_.flatten().tolist(),
                            "Intercept": model.intercept_,
                            "RMSE": root_mean_squared_error(label_test, y_pred),
                            "Spearman": spearmanr(label_test, y_pred)
                        })

                    else:
                        raise ValueError("Unknown 'task_type'. " \
                        "Available 'task_type' are 'cls' and 'reg'.")

out_df = pl.DataFrame(out)
out_df.write_parquet(OUTFN_PTH)